# Deepfake Training (ResNet-50, Xception, ViT, DFT-CNN, Canny-CNN) — Dataset 2 (Dataset.zip)
Fully ordered notebook (no missing functions). Compatible with `deepfake_detector_patched_with_mp4.py`.

Label convention: **REAL=0, FAKE=1** (matches your detector).


In [ ]:
!pip -q install timm transformers opencv-python-headless

In [ ]:
import os, random, time, json, math, zipfile, shutil, glob, pathlib
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
import timm
from transformers import ViTForImageClassification

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Dataset 2: Normalize Train/Validation/Test into train/val/test

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/Dataset.zip'  # <-- change
EXTRACT_DIR = '/content/data_src'
SPLIT_DIR = '/content/data_split'
OUT_DIR = '/content/drive/MyDrive/deepfake_checkpoints'
os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(EXTRACT_DIR): shutil.rmtree(EXTRACT_DIR)
if os.path.exists(SPLIT_DIR): shutil.rmtree(SPLIT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

def find_split(root, split_name):
    for p in pathlib.Path(root).rglob('*'):
        if p.is_dir() and p.name.lower() == split_name.lower():
            return str(p)
    return None

train_root = find_split(EXTRACT_DIR, 'Train')
val_root   = find_split(EXTRACT_DIR, 'Validation')
test_root  = find_split(EXTRACT_DIR, 'Test')
if not train_root or not val_root or not test_root:
    raise ValueError('Could not locate Train/Validation/Test in extracted folder.')

def class_dir(split_root, cls_name):
    for p in pathlib.Path(split_root).iterdir():
        if p.is_dir() and p.name.lower() == cls_name.lower():
            return str(p)
    return None

splits = {'train': train_root, 'val': val_root, 'test': test_root}
for split, src_root in splits.items():
    for cls_lc, cls_src in {'real':'Real','fake':'Fake'}.items():
        src = class_dir(src_root, cls_src)
        if not src:
            raise ValueError(f'Missing {cls_src} under {src_root}')
        dst = os.path.join(SPLIT_DIR, split, cls_lc)
        os.makedirs(dst, exist_ok=True)
        for f in glob.glob(os.path.join(src, '*')):
            if os.path.isfile(f):
                shutil.copy2(f, os.path.join(dst, os.path.basename(f)))

print('Normalized dataset ready at:', SPLIT_DIR)


## Transforms

In [ ]:
resnet_tf_train = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.1,0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])
resnet_tf_eval = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

xception_tf_train = transforms.Compose([
    transforms.Resize((299,299)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.1,0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])
xception_tf_eval = transforms.Compose([
    transforms.Resize((299,299)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

vit_tf_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.1,0.1,0.1),
    transforms.ToTensor(),
])
vit_tf_eval = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])


## Dataset wrappers enforcing REAL=0, FAKE=1

In [ ]:
class MappedImageFolder(Dataset):
    def __init__(self, root, transform):
        self.base = datasets.ImageFolder(root, transform=transform)
        self.idx_to_class = {v: k.lower() for k, v in self.base.class_to_idx.items()}
        self.desired = {'real': 0, 'fake': 1}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y_old = self.base[idx]
        cls = self.idx_to_class[int(y_old)]
        y = self.desired[cls]
        return x, y

    @property
    def targets(self):
        return [self.desired[self.idx_to_class[int(y_old)]] for _, y_old in self.base.samples]

def make_rgb_loaders(kind: str, batch=16, workers=2):
    if kind=='resnet':
        tr_tf, ev_tf = resnet_tf_train, resnet_tf_eval
    elif kind=='xception':
        tr_tf, ev_tf = xception_tf_train, xception_tf_eval
    elif kind=='vit':
        tr_tf, ev_tf = vit_tf_train, vit_tf_eval
    else:
        raise ValueError(kind)

    train_ds = MappedImageFolder(os.path.join(SPLIT_DIR,'train'), tr_tf)
    val_ds   = MappedImageFolder(os.path.join(SPLIT_DIR,'val'),   ev_tf)
    test_ds  = MappedImageFolder(os.path.join(SPLIT_DIR,'test'),  ev_tf)

    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch, shuffle=False, num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch, shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds


## DFT/Canny feature dataset + loaders

In [ ]:
class OneChannelFeatureDataset(Dataset):
    def __init__(self, root, split, feature: str):
        self.feature = feature.lower()
        self.ds = datasets.ImageFolder(os.path.join(root, split), transform=None)
        self.idx_to_class = {v: k.lower() for k, v in self.ds.class_to_idx.items()}
        self.desired = {'real': 0, 'fake': 1}

    def __len__(self):
        return len(self.ds)

    def _canny(self, bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 100, 200)
        edges = cv2.resize(edges, (256,256))
        return (edges.astype(np.float32) / 255.0)

    def _dft(self, bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        dft = cv2.dft(np.float32(gray), flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        mag = 20 * np.log(cv2.magnitude(dft_shift[:,:,0], dft_shift[:,:,1]) + 1)
        mag = cv2.resize(mag, (256,256))
        mag = cv2.normalize(mag, None, 0, 1, cv2.NORM_MINMAX)
        return mag.astype(np.float32)

    def __getitem__(self, idx):
        path, y_old = self.ds.samples[idx]
        cls = self.idx_to_class[int(y_old)]
        y = float(self.desired[cls])

        img = cv2.imread(path)
        if img is None:
            raise ValueError(f'Failed to read image: {path}')

        if self.feature == 'canny':
            feat = self._canny(img)
        elif self.feature == 'dft':
            feat = self._dft(img)
        else:
            raise ValueError(self.feature)

        x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor([y], dtype=torch.float32)
        return x, y

    @property
    def targets(self):
        return [self.desired[self.idx_to_class[int(y_old)]] for _, y_old in self.ds.samples]

def make_feature_loaders(feature: str, batch=32, workers=2):
    train_ds = OneChannelFeatureDataset(SPLIT_DIR, 'train', feature)
    val_ds   = OneChannelFeatureDataset(SPLIT_DIR, 'val',   feature)
    test_ds  = OneChannelFeatureDataset(SPLIT_DIR, 'test',  feature)

    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch, shuffle=False, num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch, shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds


## Model builders

In [ ]:
def build_resnet50_3channel():
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m

def build_xception_3channel():
    m = timm.create_model('xception', pretrained=True, num_classes=1000)
    num_ftrs = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 1),
    )
    return m

def build_vit_3channel():
    m = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    in_f = m.classifier.in_features
    m.classifier = nn.Linear(in_f, 1)
    return m

class Simple1CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128*32*32, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 128*32*32)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

def build_dft_cnn(): return Simple1CNN()
def build_canny_cnn(): return Simple1CNN()


## Train / Eval

In [ ]:
@torch.no_grad()
def eval_binary(model, loader, kind, threshold=0.5):
    model.eval()
    total=0; correct=0; loss_sum=0.0
    tp=tn=fp=fn=0
    for x,y in loader:
        x = x.to(device)
        y = y.float().to(device).view(-1,1)
        if kind=='vit':
            logits = model(pixel_values=x).logits
        else:
            logits = model(x)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y)
        loss_sum += loss.item()*y.size(0)

        p = torch.sigmoid(logits)
        pred = (p >= threshold).float()

        correct += (pred==y).sum().item()
        total += y.size(0)
        tp += ((pred==1)&(y==1)).sum().item()
        tn += ((pred==0)&(y==0)).sum().item()
        fp += ((pred==1)&(y==0)).sum().item()
        fn += ((pred==0)&(y==1)).sum().item()
    return {'loss': loss_sum/max(total,1), 'acc': correct/max(total,1),
            'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn)}

def sweep_threshold(model, loader, kind):
    best_t=0.5; best_acc=-1
    for t in np.linspace(0.05,0.95,19):
        m = eval_binary(model, loader, kind, float(t))
        if m['acc']>best_acc:
            best_acc=m['acc']; best_t=float(t)
    return best_t, best_acc

def train_one(kind, epochs=8, batch=16, lr=2e-4, wd=1e-4, amp=True):
    if kind in ['resnet','xception','vit']:
        tr, va, te, train_ds = make_rgb_loaders(kind, batch=batch)
        if kind=='resnet':
            model = build_resnet50_3channel()
            head_params = list(model.fc.parameters())
            backbone_params = [p for n,p in model.named_parameters() if not n.startswith('fc.')]
        elif kind=='xception':
            model = build_xception_3channel()
            head_params = list(model.fc.parameters())
            backbone_params = [p for n,p in model.named_parameters() if not n.startswith('fc.')]
        else:
            model = build_vit_3channel()
            head_params = list(model.classifier.parameters())
            backbone_params = [p for n,p in model.named_parameters() if not n.startswith('classifier.')]
    elif kind in ['dft','canny']:
        tr, va, te, train_ds = make_feature_loaders(kind, batch=batch)
        model = build_dft_cnn() if kind=='dft' else build_canny_cnn()
        head_params = list(model.parameters())
        backbone_params = []
    else:
        raise ValueError(kind)

    model.to(device)

    targets = np.array(train_ds.targets)
    pos = (targets==1).sum()
    neg = (targets==0).sum()
    pos_weight = torch.tensor([neg/max(pos,1)], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    params = [{'params': backbone_params, 'lr': lr*0.3}, {'params': head_params, 'lr': lr}] if backbone_params else [{'params': head_params, 'lr': lr}]
    opt = optim.AdamW(params, weight_decay=wd)
    scaler = torch.cuda.amp.GradScaler(enabled=amp)

    best_acc=-1
    best_path=None

    for ep in range(1, epochs+1):
        model.train()
        run=0.0; n=0
        for x,y in tr:
            x = x.to(device, non_blocking=True)
            y = y.float().to(device).view(-1,1)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=amp):
                if kind=='vit':
                    logits = model(pixel_values=x).logits
                else:
                    logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run += loss.item()*y.size(0); n += y.size(0)

        val = eval_binary(model, va, kind, 0.5)
        print(f"Epoch {ep}/{epochs} | train_loss={run/max(n,1):.4f} | val_loss={val['loss']:.4f} | val_acc={val['acc']:.4f} (tp={val['tp']} tn={val['tn']} fp={val['fp']} fn={val['fn']})")

        if val['acc']>best_acc:
            best_acc=val['acc']
            stamp=time.strftime('%Y%m%d_%H%M%S')
            best_path=os.path.join(OUT_DIR, f"{kind}_best_{stamp}.pth")
            torch.save(model.state_dict(), best_path)
            print('  Saved ->', best_path)

    best_t, best_t_acc = sweep_threshold(model, va, kind)
    test05 = eval_binary(model, te, kind, 0.5)
    testbt = eval_binary(model, te, kind, best_t)
    print('\n====', kind, '====')
    print('best_val_acc:', best_acc)
    print(f'best_threshold: {best_t:.2f} (val_acc={best_t_acc:.4f})')
    print('test@0.50:', test05)
    print('test@best_t:', testbt)

    if best_path:
        meta_path = best_path.replace('.pth','.meta.json')
        with open(meta_path,'w') as f:
            json.dump({'model_kind':kind,'best_threshold':best_t,'val_best_acc':best_acc}, f, indent=2)
        print('  Meta ->', meta_path)

    return best_path


## Train all models

In [ ]:
resnet_pth   = train_one('resnet',  epochs=8, batch=16, lr=2e-4)
xception_pth = train_one('xception',epochs=8, batch=12, lr=2e-4)
vit_pth      = train_one('vit',     epochs=8, batch=12, lr=2e-4)

dft_pth      = train_one('dft',     epochs=10, batch=32, lr=3e-4)
canny_pth    = train_one('canny',   epochs=10, batch=32, lr=3e-4)

print('\nSaved checkpoints:')
print(resnet_pth, xception_pth, vit_pth, dft_pth, canny_pth)
